<a href="https://colab.research.google.com/github/PHANTOMVOID0/Image-Upscaler/blob/main/Image_Upscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Image Upscaler

In [ ]:
#@title Cell 1: Install Real-ESRGAN & Optional Drive Mount

import os
from datetime import datetime

# ============ DRIVE OPTIONS ============
mount_drive = True  #@param {type:"boolean"}

DRIVE_MOUNTED = False
DRIVE_OUTPUT = None

if mount_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive')

        # Create folder structure
        DRIVE_BASE = "/content/drive/MyDrive/image upscaled"
        TODAY = datetime.now().strftime("%Y-%m-%d")
        DRIVE_OUTPUT = os.path.join(DRIVE_BASE, TODAY)

        if not os.path.exists(DRIVE_BASE):
            os.makedirs(DRIVE_BASE)
            print(f"✅ Created main folder: image upscaled/")
        else:
            print(f"✅ Found main folder: image upscaled/")

        if not os.path.exists(DRIVE_OUTPUT):
            os.makedirs(DRIVE_OUTPUT)
            print(f"✅ Created today's folder: {TODAY}/")
        else:
            print(f"✅ Today's folder exists: {TODAY}/")

        DRIVE_MOUNTED = True
        print(f"\n📁 Results will be saved to:")
        print(f"   {DRIVE_OUTPUT}\n")
    except Exception as e:
        print(f"\033[91m❌ Drive mount failed: {e}\033[0m")
        print("⚠️  Continuing without Drive - results will be local only\n")
else:
    print("ℹ️  Drive not mounted - results will be local only\n")

# Clone Real-ESRGAN
%cd /content/
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

# Install dependencies
print("⏳ Installing dependencies...")
!pip install -q facexlib
!pip install -q gfpgan
!pip install -q -r requirements.txt
!pip install -q basicsr-fixed
!python setup.py develop

# Download ALL models
print("\n📦 Downloading models (this may take 1-2 minutes)...\n")

print("⏳ [1/2] Downloading RealESRGAN_x4plus (official)...")
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P experiments/pretrained_models



print("⏳ [2/2] Downloading AESRGAN (best for faces)...")
!wget -q https://github.com/stroking-fishes-ml-corp/A-ESRGAN/releases/download/v1.0.0/A_ESRGAN_Single.pth -P experiments/pretrained_models

# Verify installation
from IPython.display import clear_output, HTML
clear_output()

print("\033[92m✅ Real-ESRGAN Installation successful\033[0m")
if DRIVE_MOUNTED:
    print(f"\033[92m✅ Drive connected: {DRIVE_OUTPUT}\033[0m\n")
else:
    print(f"\033[93m⚠️  Drive not connected (local only)\033[0m\n")

# Check all models
models = [
    ("RealESRGAN_x4plus", "experiments/pretrained_models/RealESRGAN_x4plus.pth", "Good - Fast, reliable"),
    ("UltraSharp V2", "experiments/pretrained_models/4xUltraSharp.pth", "Better - More detail, same speed"),
    ("AESRGAN", "experiments/pretrained_models/A_ESRGAN_Single.pth", "Best - Faces/portraits, slower")
]

print("📦 Available models:\n")
for name, path, description in models:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"   \033[92m✅ {name:<20} ({size_mb:5.1f}MB)\033[0m - {description}")
    else:
        print(f"   \033[91m❌ MISSING: {name}\033[0m")

# Keep Alive - Using HTML audio (doesn't block cell completion)
display(HTML('''
<audio loop autoplay controls>
  <source src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" type="audio/mp4">
</audio>
'''))

print("\n🎵 Keep-alive enabled (prevents disconnection)")
print("\n" + "="*60)
print("💡 QUICK GUIDE:")
print("   • RealESRGAN_x4plus: Best for general use (fastest)")
print("   • UltraSharp V2: Best for compressed/detailed images")
print("   • AESRGAN: Best for portraits/faces (2x slower)")
print("="*60)


In [ ]:
#@title Cell 2: Upload Images

import os
from google.colab import files
import shutil
import re
import zipfile

upload_folder = 'upload'
result_folder = 'results'

# Clean up previous folders
if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)
os.mkdir(upload_folder)
os.mkdir(result_folder)

# ============ FILE PATH (Colab or Drive) ============
file_path = ""  #@param {type:"string"}

# Supported image formats
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif'}

def extract_images_from_zip(zip_path, destination):
    """Extract only images from ZIP file"""
    extracted_count = 0
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_info in zip_ref.filelist:
            filename = file_info.filename
            if file_info.is_dir():
                continue

            ext = os.path.splitext(filename.lower())[1]
            if ext in IMAGE_EXTENSIONS:
                file_data = zip_ref.read(filename)
                safe_name = os.path.basename(filename)
                safe_name = re.sub(r'[^\w\.-]', '_', safe_name)

                output_path = os.path.join(destination, safe_name)
                with open(output_path, 'wb') as f:
                    f.write(file_data)
                extracted_count += 1
    return extracted_count

def sanitize_filename(filename):
    """Remove spaces and special characters"""
    safe_name = re.sub(r'[^\w\.-]', '_', filename)
    if '.' not in safe_name:
        safe_name += '.jpg'
    return safe_name

print(f"{'='*50}")

# Check if file path is provided
if file_path.strip():
    # User provided a path (Colab or Drive)
    print("📁 Loading from provided path...")

    if not os.path.exists(file_path):
        print(f"\033[91m❌ ERROR: File not found at:\033[0m")
        print(f"   {file_path}\n")
        print("💡 Examples:")
        print("   /content/demo.zip (file uploaded to Colab)")
        print("   /content/drive/MyDrive/demo.zip (file in Drive)")
    else:
        ext = os.path.splitext(file_path.lower())[1]

        if ext == '.zip':
            print(f"📦 Extracting images from ZIP...")
            count = extract_images_from_zip(file_path, upload_folder)
            print(f"   ✅ Extracted {count} images")
        else:
            safe_name = sanitize_filename(os.path.basename(file_path))
            dst_path = os.path.join(upload_folder, safe_name)
            shutil.copy2(file_path, dst_path)
            print(f"   ✅ Copied: {safe_name}")

else:
    # No path provided, use upload button
    print("📂 Upload your images or ZIP file:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        file = filename
        ext = os.path.splitext(filename.lower())[1]

        if ext == '.zip':
            print(f"\n📦 Extracting images from: {filename}")
            count = extract_images_from_zip(file, upload_folder)
            print(f"   ✅ Extracted {count} images")
            os.remove(file)
        else:
            safe_name = sanitize_filename(filename)
            dst_path = os.path.join(upload_folder, safe_name)
            shutil.move(file, dst_path)
            print(f'   ✅ Renaming "{filename}" to "{safe_name}"')

# Show final list
total_images = len([f for f in os.listdir(upload_folder) if os.path.isfile(os.path.join(upload_folder, f))])
print(f"\n{'='*50}")
print(f"✅ Total images ready: {total_images}")
print(f"✅ Files ready for processing:")
for f in os.listdir(upload_folder):
    print(f"   📷 {f}")
print(f"{'='*50}")


In [ ]:
#@title Cell 3: Upscale Images with Real-Time Drive Backup


# ============ MODEL SELECTION ============
model_choice = "AESRGAN" #@param ["RealESRGAN_x4plus", "AESRGAN"]


MODEL_PATHS = {
    "RealESRGAN_x4plus": "experiments/pretrained_models/RealESRGAN_x4plus.pth",
    "AESRGAN": "experiments/pretrained_models/A_ESRGAN_Single.pth"
}


selected_model = MODEL_PATHS[model_choice]


# ============ UPSCALE SETTINGS ============
outscale = 4 #@param {type:"slider", min:2, max:4, step:1}
tile_size = 1024 #@param {type:"slider", min:128, max:1024, step:64}
tile_pad = 10 #@param {type:"slider", min:5, max:20, step:5}
face_enhance = False #@param {type:"boolean"}
output_format = "png" #@param ["png", "jpeg", "jpg", "webp"]


# ============ FILE SIZE CONTROL ============
jpg_quality = 100 #@param {type:"slider", min:70, max:100, step:5}
min_file_size_mb = 4 #@param {type:"slider", min:0, max:20, step:1}


import time
import shutil
import os
from datetime import datetime
from PIL import Image


# Check if Drive is mounted
DRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')
DRIVE_OUTPUT = None


if DRIVE_MOUNTED:
    DRIVE_OUTPUT = os.path.join("/content/drive/MyDrive/image upscaled", datetime.now().strftime("%Y-%m-%d"))
    if not os.path.exists(DRIVE_OUTPUT):
        os.makedirs(DRIVE_OUTPUT)


print(f"{'='*50}")
print(f"🤖 Model: {model_choice}")
print(f"🎯 Upscale Factor: {outscale}x")
print(f"🧩 Tile Size: {tile_size}px")
print(f"✨ Face Enhancement: {'Enabled' if face_enhance else 'Disabled'}")
print(f"📄 Output Format: {output_format.upper()}")
print(f"⚖️  Min File Size: {min_file_size_mb} MB")
if output_format in ["jpg", "jpeg", "webp"]:
    print(f"🎨 Quality: {jpg_quality}%")
if DRIVE_MOUNTED:
    print(f"☁️  Real-time Drive backup: Enabled")
    print(f"📁 Drive path: {DRIVE_OUTPUT}")
else:
    print(f"💾 Save location: Local only (Drive not mounted)")
print(f"{'='*50}\n")


# Get list of all input images
input_images = sorted([f for f in os.listdir(upload_folder) if os.path.isfile(os.path.join(upload_folder, f))])
total_images = len(input_images)
print(f"📝 Total images to process: {total_images}\n")


# Clear results folder before processing
if os.path.exists(result_folder):
    shutil.rmtree(result_folder)
os.makedirs(result_folder)


if total_images == 0:
    print("\033[91m❌ ERROR: No images found in upload folder!\033[0m")
else:
    # Process each image individually
    batch_start_time = time.time()


    for idx, image_name in enumerate(input_images, 1):
        print(f"🔄 Processing image {idx}/{total_images}: {image_name}")
        image_start_time = time.time() # Start timer for this image


        # Build command for single image
        input_path = os.path.join(upload_folder, image_name)
        base_name = os.path.splitext(image_name)[0]


        # Output directly to PNG with NO suffix
        cmd = f"""python inference_realesrgan.py \
            --model_path {selected_model} \
            -i "{input_path}" \
            -o results \
            -s {outscale} \
            -t {tile_size} \
            --tile_pad {tile_pad} \
            --ext png \
            --suffix ''"""


        if face_enhance:
            cmd += " --face_enhance"


        # Process this image
        !{cmd}


        # Wait for file system to update
        time.sleep(0.5)


        # ============ FIXED: PRESERVE EXACT FILENAME ============
        # Real-ESRGAN might have changed special characters like (3) to _3_
        # Find the actual output file it created
        existing_files_before = set(os.listdir(result_folder)) if idx > 1 else set()
        time.sleep(0.3)
        existing_files_after = set(os.listdir(result_folder))
        new_files = existing_files_after - existing_files_before

        # Get the newly created file
        if new_files:
            actual_output_file = list(new_files)[0]
            temp_png = os.path.join(result_folder, actual_output_file)
        else:
            # Fallback: try expected name
            temp_png = os.path.join(result_folder, f"{base_name}.png")


        # Final output name (EXACT base name + new extension)
        output_name = f"{base_name}.{output_format}"
        output_path = os.path.join(result_folder, output_name)


        # Convert to desired format with quality control
        if os.path.exists(temp_png):
            img = Image.open(temp_png)


            if output_format in ["jpg", "jpeg"]:
                # Save as JPEG with maximum quality settings
                img.convert('RGB').save(output_path, 'JPEG', quality=jpg_quality, optimize=False, subsampling=0)


            elif output_format == "png":
                # Save PNG without compression for maximum size
                img.save(output_path, 'PNG', optimize=True, compress_level=0)


            elif output_format == "webp":
                # Save as WebP
                img.save(output_path, 'WEBP', quality=jpg_quality, method=0)


            # Remove temporary PNG (only if we converted to another format)
            if os.path.exists(temp_png) and temp_png != output_path:
                os.remove(temp_png)


            # ==============================================================================
            # 🛡️ MINIMUM FILE SIZE ENFORCEMENT (User Controlled)
            # ==============================================================================
            if min_file_size_mb > 0:
                target_size_bytes = min_file_size_mb * 1024 * 1024


                # Check for modern Pillow version
                resample_method = getattr(Image, 'Resampling', Image).LANCZOS


                loop_count = 0
                while os.path.getsize(output_path) < target_size_bytes and loop_count < 2:
                    current_mb = os.path.getsize(output_path) / (1024 * 1024)
                    print(f"   🔸 Size {current_mb:.2f}MB < {min_file_size_mb}MB. Lanczos 25% upscale...")


                    # Re-open the file
                    img = Image.open(output_path)
                    cur_w, cur_h = img.size


                    # Calculate new dimensions (+25%)
                    new_w = int(cur_w * 1.25)
                    new_h = int(cur_h * 1.25)


                    # Apply High-Quality Lanczos Filter
                    img = img.resize((new_w, new_h), resample_method)


                    # Save again (Overwrite)
                    if output_format in ["jpg", "jpeg"]:
                        img.convert('RGB').save(output_path, 'JPEG', quality=jpg_quality, optimize=False, subsampling=0)
                    elif output_format == "png":
                        img.save(output_path, 'PNG', optimize=True, compress_level=0)
                    elif output_format == "webp":
                        img.save(output_path, 'WEBP', quality=jpg_quality, method=0)


                    loop_count += 1
            # ==============================================================================


            # Gather Statistics
            image_duration = time.time() - image_start_time


            # Get Input Stats
            with Image.open(input_path) as img_in:
                in_w, in_h = img_in.size
                in_mp = (in_w * in_h) / 1_000_000
                input_size_kb = os.path.getsize(input_path) / 1024


            # Get Output Stats
            with Image.open(output_path) as img_out:
                out_w, out_h = img_out.size
                out_mp = (out_w * out_h) / 1_000_000
                output_size_mb = os.path.getsize(output_path) / (1024 * 1024)


            # Print Report
            print(f"⏱️ Time: {image_duration:.2f}s")
            print(f"📐 Input:  {in_w}x{in_h} ({in_mp:.2f} MP) • {input_size_kb:.1f}KB")
            print(f"📏 Output: {out_w}x{out_h} ({out_mp:.2f} MP) • {output_size_mb:.2f}MB")


            if DRIVE_MOUNTED:
                drive_path = os.path.join(DRIVE_OUTPUT, output_name)
                shutil.copy2(output_path, drive_path)
                print(f"✅ Image {idx}/{total_images} processed ✅ Saved to Drive ✅\n")
            else:
                print(f"✅ Image {idx}/{total_images} processed ✅\n")
        else:
            print(f"⚠️  Warning: Output not found for {image_name}\n")


    elapsed = time.time() - batch_start_time


    # Count final results
    processed_count = len([f for f in os.listdir(result_folder) if os.path.isfile(os.path.join(result_folder, f))])


    # Create and upload ZIP
    print(f"{'='*50}")
    if processed_count > 0:
        print(f"📦 Creating ZIP file...")
        zip_filename = f'/content/Real-ESRGAN_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'
        !zip -r -j -q {zip_filename} results/*


        # Upload ZIP to Drive
        if DRIVE_MOUNTED:
            zip_drive_path = os.path.join(DRIVE_OUTPUT, os.path.basename(zip_filename))
            shutil.copy2(zip_filename, zip_drive_path)
            print(f"☁️  ZIP also uploaded to Drive")


        print(f"\n🎉 COMPLETED!")
        print(f"✅ Processed: {processed_count} images")
        print(f"⏱️  Total time: {elapsed/60:.2f} minutes")
        print(f"⚡ Speed: {elapsed/processed_count:.2f} sec/image")
        print(f"🤖 Model used: {model_choice}")


        if DRIVE_MOUNTED:
            print(f"\n📁 Drive location: {DRIVE_OUTPUT}")


        # Download ZIP
        from google.colab import files
        files.download(zip_filename)
        print(f"\n⬇️  Local ZIP download started!")
        print(f"\033[92m✅ All done!\033[0m")
    else:
        print(f"❌ ERROR: No results generated!")


    print(f"{'='*50}")
